In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [84]:
df = pd.read_excel('/content/raw_data.xlsx')

In [85]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       1200 non-null   object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [87]:
df.size

16800

In [88]:
df.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [89]:
df.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

# CHECK MISSING VALUES


In [90]:
df.isnull().sum()

,0
OrderID,0
Date,0
CustomerID,0
Product,0
Quantity,0
UnitPrice,0
ShippingAddress,0
PaymentMethod,0
OrderStatus,0
TrackingNumber,0


CHECK DUPLICATE ROWS

In [91]:
df.duplicated().sum()

np.int64(0)

HANDLE MISSING VALUES

In [92]:
numeric_cols = df.select_dtypes(include=np.number).columns

In [93]:
for col in numeric_cols:
  df[col] = df[col].fillna(df[col].median())

In [94]:
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
  df[col] = df[col].fillna(df[col].mode()[0])

REMOVE DUPLICATES

In [95]:
df.drop_duplicates(inplace = True)

STANDARDIZE COLUMN NAMES

In [96]:
df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_")
df.columns

Index(['orderid', 'date', 'customerid', 'product', 'quantity', 'unitprice',
       'shippingaddress', 'paymentmethod', 'orderstatus', 'trackingnumber',
       'itemsincart', 'couponcode', 'referralsource', 'totalprice'],
      dtype='object')

CLEAN TEXT DATA

In [97]:
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
for col in categorical_cols:
  df[col] = df[col].astype(str).str.strip().str.title()

DATE CONVERSION

In [98]:
for col in df.columns:
  if 'date' in col:
    df[col] = pd.to_datetime(df[col], errors='coerce')

REMOVE INVALID VALUES


In [99]:
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
  df = df[df[col] >= 0]

VERIFY TOTAL PRICE CALCULATION

In [100]:
df['calculated_total'] = df['quantity'] * df['unitprice']

In [101]:
comparison = df['totalprice'] == df['calculated_total']
print("\nCorrect TotalPrice Rows:", comparison.sum())
df['totalprice'] = df['calculated_total']
df.drop(columns=['calculated_total'], inplace=True)


Correct TotalPrice Rows: 1093


FINAL DATASET CHECK

In [102]:
df.isnull().sum()

,0
orderid,0
date,0
customerid,0
product,0
quantity,0
unitprice,0
shippingaddress,0
paymentmethod,0
orderstatus,0
trackingnumber,0


In [103]:
df.dtypes

,0
orderid,object
date,datetime64[ns]
customerid,object
product,object
quantity,int64
unitprice,float64
shippingaddress,object
paymentmethod,object
orderstatus,object
trackingnumber,object


In [104]:
df.shape

(1200, 14)

SAVE CLEANED DATASET

In [105]:
df.to_excel("cleaned_dataset.xlsx", index=False)

In [106]:
from google.colab import files

files.download("cleaned_dataset.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>